# DINOv2: Learning Robust Visual Features without Supervision
**Authors:** Maxime Oquab, Timothée Darcet, Théo Moutakif, et al. (Meta AI, 2023)

## 1. Data-Pipeline in Details

### 1.1 Data-Pipeline
The DinoV2 teams was able to create a huge semi-curated dataset called **LVD-142M**. 

> I am calling it semi-curated, since it is using a pre-trained model on a set of curated data to curate the rest of the uncurated 1.2B images.

The pipeline consists of 2 general sources:
- Curated data: It contains various human-labeled curated datasets like (ImageNet-22k, Mapillary SLS, Google Landmarks v2 etc.)
- Uncurated web data: It contains 1,3 Billion web-scraped images. Random, noise messy data.

The web data was acquired by scraping url links of images from a publicly available repository of crawled web data. It got the `<img>` tags and filtered for unsafe/restricted domains (dedup, nsfw, blurring etc.). 

<img src="Images/data_pipeline_curation.png" width=800>

The goal was to expand the curated data with the uncurated data in a good/reliable way.

Their pipeline does the following:
#### 1. Pre-training a model for comparisson
The first thing they needed to do was pre-train a model used for the embedding and retrieval process. It was a ViT-H/16 network that had been pre-trained specifically on the ImageNet-22k dataset.

Instead of using a model trained on all curated data, the authors started with a model trained on the largest and most diverse single curated source they had: **ImageNet-22k**.

#### 2.Embedding the images: 
The second step was to map all images (curated and uncurated) into embeddings with the pre-trained model. This way we have a mathematical representation of all the images. Now we can compare them!

#### 3.Deduplication: 
This step is about efficiency and removing redundancy within the 1.3B image web scrape itself.

##### 3.1 Image similarity
They employ cosine similarity to compare image features with the following similarity function m:

$$ m(s, r) = \text{cosine-similarity}(f(s), f(r)) = \frac{f(s) \cdot f(r)}{\|f(s)\|_2 \|f(r)\|_2} $$

where $ s $ and $ r $ are a pair of images to compare and $ f $ is the model generating features (the pre-trained model)

##### 3.2 Self-Deduplication (Internal Cleaning)
The goal was to ensure the model doesn't waste capacity or training time looking at the exact same image (or nearly identical ones, like stock photos or re-posts) millions of times.

To do this they compute and use the generated embeddings to retrieve the k = 64 nearest neighbors of each image (using cosine similarity). Considering only neighbors with a similarity $ >0.6 $, they then only keep one representative for each class of duplicate images. This results in a self-deduplicated data source of **1.1B** images.

##### 3.3 Relative deduplication (Benchmark Protection)

The idea is to remove any images from the web scrape that are too similar to the images in the evaluation datasets (the benchmarks used to test the model later). It uses a stricter similarity score of $ >0.45 $. Because this threshold is lower, it catches images that are even "vaguely" similar to the test sets, not just exact copies. Instead of keeping one representative, it discards the entire class if it matches a reference image from a benchmark.

This results in a relatively-deduplicated data source of **744M** images from the previous: **1.1B** 

#### 4. Matching/Retrieval (Broader Threshold): 
Matches the curated images to the uncurated images using the pre-trained model based on semantic similarity, not visual identity. If the curated set has a dog, it pulls different photos of dogs (new poses, lighting, backgrounds) from the web data.

This part describes how they "grow" the smaller curated datasets into massive, 1-million-image chunks by retrieving them from the 1.2 billion image web pool.

They employ two different strategies depending on how big the starting (seed) dataset is:

##### 4.1. Sample-based
Sample-based, applies to datasets larger than 1M images and consists in collecting a fixed number k of nearest images for each sample image to retrieve. K is the balancing factor of the final dataset. How much of a representation do they want each dataset to have in the end.

- They use `k = 4` for `Google Landmarks v2` and `ImageNet-22k`
    - <img src="Images/google_landmarks.png" width = 800>
    - <img src="Images/image_net.png" width = 800>
- But a larger `k = 32` for `ImageNet-1k` to make this specific retrieval a core part of our LVD-142M dataset.
    - <img src="Images/image_net_1k.png" width = 800>


##### 4.2. Cluster-based. 
This is used for the "Fine-Grained" datasets that are very small (sometimes only a few hundred or thousand images)
1. They use k-means clustering to group the entire **744M** web scrape into 100,000 clusters based on visual concepts.

2. Then they look at a cluster and ask: "Does this cluster contain at least 4 images from my curated seed?".

3. Finally if the answer is yes, they assume that entire bucket is "relevant" to that concept and pull up to 10,000 images from it.

As this can result in a very large number of retrieved images for some dataset, they restrict such retrievals to a maximum of **1M** images to maintain the balance between the different datasets within LVD-142M
#### 5. Merge: 
By merging the original curated images with the matched web images, they create a vastly expanded dataset of 142,109,386 images.

It takes less than two days to produce the LVD-142M dataset

### 1.2 Dataset
The researches started with gathering a widespread amount of datasets that were human-labeled (hence curated) as the ground truth for the data.

In the following picture we can see a list of all of them that they used in the curated set under the **Dataset** column:

<img src="Images/composition_of_dataset.png" width=800>

Notable things to see from this table:
- There are are very diverse set of datasets like:
    - Flowers
    - Stanford Cars
    - Cituscapes
    - etc.
- The biggest notable datasets are:
    - **ImageNet-22k:** 14,197,086 images
    - **ImageNet-1k:** 1,281,167 images
    - **Mapillary SLS:** 1,434,262 images
    - **Google Landmarks v2:**  1,580,470 images
- **ImageNet-22k** and **ImageNet-1k** are included in final retrieval almost with the same resulting weight. **56M** against **41M**. They wanted to have a big representation of images from both of them.
- All of the small datasets that are 1k/2k/5k are included with exactly **1M** similar images.

The idea was to have a balanced dataset which represents many different objects. This is the reason why the model was able to perform so well on all the various tasks with just a linear probe for the specific task. Monicular detection, calssification, segmentation, you name it. This is where everything starts. Because of this engineering pre-work.

## 2. Role of resolution

> *Explain one important figure or table

<img src="Images/resolution.png" width=1000>

We are seeing a graph that tracks the performance of 3 different models on 2 different datasets.

1. ImageNet-1k - is a foundational computer vision dataset containing ~1.28 million training images, 50,000 validation images, and 100,000 test images spanning 1,000 object categories. It is the standard benchmark for training and evaluating classification models

2. ADE-20K - ADE20K is a large-scale dataset used primarily for semantic segmentation and scene parsing, containing over 27,000 images annotated with pixel-level precision. It is widely used to train and benchmark computer vision models for tasks like instance segmentation and object detection

High-resolution training is compute-intensive, so they conducted this ablation on a small setup: a **ViT-L/16** trained on **ImageNet1k**

They pre-trained the **ViT-L/16** only on the images from **ImageNet1k** in 3 different ways:
- One model on 224x224 resolution images - the Yellow line
- One model on 416x416 resolution images - the Purple? line
- One model on 224x224 initially, then for 10k steps on 416x416 resolution images - the blue line

Then they freeze the weights of all 3 and train them on the LABELS of **ImageNet1k** and perform the tests above:
1. Classification **ImageNet-1k**, on the left
    - On the Y axis we see **Accuracy**
    - On the X axis we see resolution of images it was tested with

2. Segmentation **ADE-20K**, on the right
    - <img src="Images/iou.png" width=400>
    - On the Y axis we see **mIoU** (mean Intersection over Union) - is a metric that tracks how much 2 areas overlap, since segmentation is classes over pixel areas
    - On the X axis we see resolution of images it was tested with

In both cases we see a discrete graph, meaning they tested with 8 different resolutions, spanning from ~120x120 to ~770x770. Hard to see from the graph. Tabular data isnt available.

We see similar behaviour on both tasks. The weaker model trained on 224x224 (yellow) is performing well initially, just like the others up to its trained resolution 224x224 or a bit over it 336x336. 

As soon as the resolution climbs to higher rates (> 336x336), we see its performance drop significantly. While the other 2 models, one trained on 416x416 and the one trained on 224x224 then moved to 416x416 for 10k iterations, keep performing upward and stay very closely to one another.

All the variables for training and training methods (schedulers etc.) were kept the same to isolate the difference in resolution training. This tells us that indeed training a model on a lower resolution initially, and then climbing to a higher resolution performs almost as well as a model that was pre-trained fully on a higher resolution.

### Why is this important?
Training foundational models takes a lof of resources. Time and money for the compute. This shows that very similar results can be achieved with this cheaper/faster method of training on lower resolution first(to potentially get general understanding of visual features | my understanding) and then moving to righer resolution training (to potentially learn details | my understanding)

Quote:
> On the other hand, training at high resolution for only 10k iterations at the end of the training is almost as good and only requiring a fraction of the compute.As a consequence, we include this step at the end of the training rather than training at a high resolution from scratch.

this supports the claim that:

> Increasing image resolution is key to pixel level downstream tasks such as segmentation or detection, where small objects disappear at low resolutions. However, training at high resolution is time and memory demanding, and instead, we increase the resolution of images to 518×518 during a short period at the end of pretraining